# Generate tables for the empirical assessment on P1000 matched somatic +/- germline data at the patient-level
Here, we analyze results of running models on empirical data: matched somatic +/- germline data from the P1000 dataset.

Prerequisites: 
- you ran the empirical assessment experiment over some number of independent, stratified train/val/test splits
- the results are saved as a CSV (you can generate this by running the relevant portion of the `notebooks/analysis/make_supplementary_tables.ipynb` notebook)

In [ ]:
# Import Required Libraries
import logging
import sys

import pandas as pd

from pnet.data_analysis import permutation_utils as pu

sys.path.insert(0, "../..")  # add project_config to path
import project_config

# Setup Logging and Configuration
logging.basicConfig(
    format="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

In [ ]:
# Define directories for saving results
SAVE_RESULTS_NAME = "datasplit_analysis_p1000"
SAVE_FIGS_DIR = project_config.FIGURE_3_DIR / SAVE_RESULTS_NAME
SAVE_TABLES_DIR = project_config.SUPPLEMENTARY_TABLES_DIR # / SAVE_RESULTS_NAME

## Overall plan:
[patient-level AUPRC] For each `split_id` and `model`, do `B` patient-level bootstrap resamples (sampling patient IDs with replacement from that split's test set) and compute AUPRC on each resample → get bootstrap distributions of AUPRC per split×model.

[patient-level ΔAUPRC] For ΔAUPRC between two models, form the distribution of (AUPRC_modelA − AUPRC_modelB) by subtracting paired bootstrap draws (pairing draws by bootstrap index) within each split; then concatenate these delta draws across splits to obtain an overall cohort variability distribution and percentile CI. 

Perform analogous procedure for ΔAUPRC between two input datasets.

[split-level results] Paired Wilcoxon: operate on per-split AUPRCs (one AUPRC per split per model) and calculate the two-sided Wilcoxon signed-rank test with `scipy.stats.wilcoxon`. This tests whether AUPRCs differ across data splits (captures dataset sampling variability).


Pseudocode for patient-level bootstrap CIs:
```
for each split s:
    draw B=10k patient-level bootstrapped sample sets from this split's test set (test_s)
    for each bootstrapped sample set b:
        calculate AUPRC for each model x split combination: (AUPRC_{s,m}^(b))
        calculate ΔAUPRC for each model pair x model x split combination: ΔAUPRC_{s,AB}^(b) = AUPRC_{s,A}^(b) - AUPRC_{s,B}^(b)
        calculate ΔAUPRC for each dataset pair x split combination: ΔAUPRC_{s,m,d1-d2}^(b) = AUPRC_{s,m,d1}^(b) - AUPRC_{s,m,d2}^(b)
    compute split-level 95% bootstrap confidence intervals from {AUPRC_{s,m}^(b)}_b, {ΔAUPRC_{s,AB}^(b)}_b, and {ΔAUPRC_{s,m,d1-d2}^(b)}_b. Each of these sets should be of size B.
```

Pseudocode for split-level bootstrap CIs:
```
for each split:
    get AUPRC for each model x split combination on this split's test set (test_s) (AUPRC_{s,m})
    get ΔAUPRC for each model pair x split combination ΔAUPRC_{s,AB} = AUPRC_{s,A} - AUPRC_{s,B}
    get ΔAUPRC for each dataset pair x model x split combination ΔAUPRC_{s,m,d1-d2} = AUPRC_{s,m,d1} - AUPRC_{s,m,d2}
compute 95% boostrap confidence intervals from {AUPRC_{s,m}}_s, {ΔAUPRC_{s,AB}}_s, and {ΔAUPRC_{s,m,d1-d2}}_s. Each of these sets should be of size S = 10 (the number of splits).
```


In [ ]:
logger.info("Loading patient-level predictions for P1000 10 independent datasplits experiment")
preds_long = pd.read_csv(
    project_config.SUPPLEMENTARY_TABLES_DIR
    / "p1000_predictionMetrics_per_datasplit_patient_level.csv"
)
display(preds_long)


model_pairs = [
    ("pnet", "rf"),
    ("pnet", "logistic_regression"),
    ("rf", "logistic_regression"),
]
logger.info("Model pairs to compare: \n%s", model_pairs)


dataset_pairs = pu.make_dataset_pairs_baseline_vs_many(
    preds_long["datasets"].unique(),
    germline_baseline="germline_rare_lof",
    somatic_baseline="somatic_amp somatic_del somatic_mut",
)
logger.info("Dataset pairs to compare: \n%s", dataset_pairs)

In [ ]:
df_p1, df_p2, df_p3, df_s1, df_s2, df_s3 = pu.patient_bootstrap_cis_streaming(
    preds_long=preds_long,
    model_pairs=model_pairs,
    dataset_pairs=dataset_pairs,
    n_boot=1000,
    alpha=0.05,
    random_state=0,
)


In [ ]:
SAVE_TABLES_DIR

In [ ]:
df_s1.sort_values(
    by=["datasets", "model_type"],
    ascending=[True, False]
).round(3).to_csv(
    SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC.csv", index=False)
logger.info("Saved df_s1 to %s", SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC.csv")

df_s2.sort_values(
    by=["datasets"],
    ascending=[True]
).round(3).to_csv(
    SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC_model_comparison.csv", index=False)
logger.info("Saved df_s2 to %s", SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC_model_comparison.csv")

df_s3.sort_values(
    by=["model_type"],
    ascending=[False]
).round(3).to_csv(
    SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC_dataset_comparison.csv", index=False)
logger.info("Saved df_s3 to %s", SAVE_TABLES_DIR / "p1000_datasplit_patientBootstrap_AUPRC_dataset_comparison.csv")